# Explore the platform

Five years of daily US equity data in a local Parquet lake: OHLCV bars,
per-session universe, corporate actions and FINRA short datasets. DuckDB
queries it, dbt builds derived tables. Every read goes through `sdp.dal`,
which returns a lazy `DuckDBPyRelation`. Materialise with `.pl()` or
`.fetchall()`. Each section is independent.

In [ ]:
import datetime as dt

import polars as pl

from sdp import dal

pl.Config.set_tbl_rows(15)
pl.Config.set_fmt_str_lengths(60)

con = dal.con()   # The one connection that owns every relation.
print(dal.status())

---
## 1. What is published

A partition exists only after its audit passed. Event streams show a partition
count and date span; corporate action tables show a row count.

In [ ]:
print(dal.status())

In [ ]:
# gaps() lists the XNYS sessions inside a range that have no partition.
# An empty list means the range is complete.
first, last = dal.coverage(dal.DAY_AGGS)
print("day aggregates", first, "to", last)
print("interior gaps:", dal.gaps(dal.DAY_AGGS, first, last) or "none")

---
## 2. Point-in-time

Event streams store one immutable partition per session. The corporate action
tables do not — the endpoint has no `as_of`, so they hold the vendor's present
belief. Read with `dal.current()`, `dal.splits()`, or `dal.dividends()`.

In [ ]:
splits = dal.splits()
print("split rows:", con.sql("select count(*) from splits").fetchone()[0])
print("built from vendor pull of:",
      con.sql("select distinct vendor_pull_date from splits").fetchone()[0])

dal.splits().limit(3).pl()

In [ ]:
# The two kinds refuse each other's accessor.
for call, label in [
    (lambda: dal.series(dal.SPLITS), "series() on a current-state dataset"),
    (lambda: dal.current(dal.DAY_AGGS), "current() on an event stream"),
]:
    try:
        call()
    except ValueError as exc:
        print(f"{label}:\n  {exc}\n")

### What the current table cannot tell you

A restated factor overwrites the old value — no trace in `raw/`. Recovery path:
`ca.rebuild(dataset, pull_date)` from `vendor/`, which keeps every pull dated.

In [ ]:
from sdp.ingest import massive_corporate_actions as ca

for pull, path in ca.vendor_pulls("massive_splits"):
    print(f"{pull}  {path.stat().st_size / 1e6:8.1f} MB  {path.name}")

In [ ]:
# The published table against what an older pull said. rebuild() would replace
# raw/, so read the vendor file directly here to leave the lake alone.
old_pull, old_path = ca.vendor_pulls("massive_splits")[0]
con.sql(f'''
    with older as (
        select * from read_json('{old_path}',
                                format='newline_delimited', sample_size=-1)
    )
    select
        (select count(*) from splits) as rows_now,
        (select count(*) from older)  as rows_in_the_pull_of_{old_pull:%Y_%m_%d}
''').pl()

---
## 3. Restatement

A restatement is the vendor changing what it says about the past. The factor is
cumulative, so a revision changes every adjusted price before that event.
`sdp.restatement` compares two full pulls over `vendor/`.

In [ ]:
# Register the oldest and newest kept pulls as views over vendor/.
pulls = dict(ca.vendor_pulls("massive_splits"))
old_pull, new_pull = min(pulls), max(pulls)
print(f"comparing the pull of {old_pull} against {new_pull}")

for name, pull in [("older", old_pull), ("newer", new_pull)]:
    con.execute(f"""
        create or replace temp view {name} as select * from
        read_json('{pulls[pull]}', format='newline_delimited', sample_size=-1)
    """)

con.sql("""
    select (select count(*) from older) as rows_older,
           (select count(*) from newer) as rows_newer
""").pl()

The row count moved. Try diffing on the vendor `id` first.

In [ ]:
con.sql('''
    select
        (select count(*) from older a
          where not exists (select 1 from newer b where b.id = a.id)
        ) as ids_only_in_the_old_pull,
        (select count(*) from newer b
          where not exists (select 1 from older a where a.id = b.id)
        ) as ids_only_in_the_new_pull
''').pl()

Hundreds of ids vanish — but the same event is still present under a new id.
Match on `(ticker, execution_date)` instead.

In [ ]:
con.sql('''
    with dropped as (
        select a.* from older a
        where not exists (select 1 from newer b where b.id = a.id)
    )
    select
        count(*) as ids_that_vanished,
        count(*) filter (
            exists (select 1 from newer n
                    where n.ticker = d.ticker and n.execution_date = d.execution_date)
        ) as same_event_still_present_under_a_new_id
    from dropped d
''').pl()

The vendor `id` is not stable across pulls. Use the event key. Now measure the
real change.

In [ ]:
# Restrict to unambiguous keys in both pulls, or the join fans out.
con.sql('''
    with ua as (select ticker, execution_date from older
                group by 1, 2 having count(*) = 1),
         ub as (select ticker, execution_date from newer
                group by 1, 2 having count(*) = 1),
         k  as (select * from ua intersect select * from ub)
    select
        (select count(*)
           from older a join newer b using (ticker, execution_date)
                        join k using (ticker, execution_date)
          where a.historical_adjustment_factor
                is distinct from b.historical_adjustment_factor) as factor_restated,
        (select count(*) from newer b
          where not exists (select 1 from older a
                            where a.ticker = b.ticker
                              and a.execution_date = b.execution_date)) as genuinely_new_events
''').pl()

A restated factor changes every adjusted price before that event. A study
records the `vendor_pull_date` of the table it used.

In [ ]:
# The tickers that were restated. These are the names to look at first.
con.sql('''
    with ua as (select ticker, execution_date from older
                group by 1, 2 having count(*) = 1),
         ub as (select ticker, execution_date from newer
                group by 1, 2 having count(*) = 1),
         k  as (select * from ua intersect select * from ub)
    select a.ticker, a.execution_date, a.adjustment_type,
           a.historical_adjustment_factor as factor_before,
           b.historical_adjustment_factor as factor_after
    from older a join newer b using (ticker, execution_date)
                 join k using (ticker, execution_date)
    where a.historical_adjustment_factor
          is distinct from b.historical_adjustment_factor
    order by a.execution_date desc
    limit 15
''').pl()

---
## 4. Day aggregates

Unadjusted OHLCV, one row per ticker and session. Unadjusted on purpose — an
adjusted price restates retroactively, which breaks partition immutability.

In [ ]:
bars = dal.day_aggs()
con.sql("select count(*) as rows, count(distinct ticker) as tickers, "
        "min(date) as first_session, max(date) as last_session from bars").pl()

In [ ]:
con.sql('''
    select date, count(*) as tickers,
           round(sum(volume * close) / 1e9, 1) as dollar_volume_bn
    from bars group by date order by date
''').pl()

`volume` is floating point, not integer. The consolidated tape carries
fractional share quantities.

In [ ]:
con.sql('''
    select count(*) as rows,
           count(*) filter (volume <> floor(volume)) as fractional_volume,
           round(100.0 * count(*) filter (volume <> floor(volume)) / count(*), 1) as pct
    from bars
''').pl()

---
## 5. Universe and identifiers

Two filters build the universe, both recomputed per date.

In [ ]:
names = dal.tickers_on(dal.partitions(dal.TICKERS)[-1])
con.sql('''
    select type, count(*) as n
    from names group by type order by n desc limit 12
''').pl()

In [ ]:
con.sql('''
    select
        count(*) as all_instruments,
        count(*) filter (type = 'CS') as common_stock,
        count(*) filter (type = 'CS'
                         and primary_exchange in ('XNYS','XNAS','XASE')) as after_exchange_filter
    from names
''').pl()

### The identifier problem

Ticker symbols change and get reused after delistings. `composite_figi` is the
intended key but its coverage is patchy (see below).

In [ ]:
con.sql('''
    select
        count(*) as cs_rows,
        count(*) filter (composite_figi is null) as null_composite_figi,
        count(*) filter (share_class_figi is null) as null_share_class_figi,
        count(*) filter (cik is null) as null_cik
    from names where type = 'CS'
''').pl()

In [ ]:
# FIGI is complete for ETFs, patchy for common stock. CIK is the opposite.
con.sql('''
    select type, count(*) as n,
           count(*) filter (composite_figi is null) as null_figi,
           count(*) filter (cik is null) as null_cik
    from names
    where type in ('CS', 'ETF', 'ADRC', 'WARRANT', 'PFD')
    group by type order by n desc
''').pl()

The FIGI gap after the liquidity filter decides whether the fallback matters.
That diagnostic needs the backfill (section 9).

---
## 6. Adjustment for corporate actions

The vendor's `historical_adjustment_factor` is cumulative: for a price on D,
take the first event after D and multiply. Check against AAPL (2:1 in 2005,
7:1 in 2014, 4:1 in 2020).

In [ ]:
splits = dal.splits()
con.sql('''
    select execution_date, adjustment_type,
           split_from, split_to,
           split_to / split_from as ratio,
           historical_adjustment_factor
    from splits where ticker = 'AAPL' order by execution_date
''').pl()

In [ ]:
# The 2005 factor must equal 1/2 * 1/7 * 1/4 = 1/56, compounded with the later
# two splits. Reproducing it from the ratios confirms the semantics.
expected = 1 / (2 * 7 * 4)
print(f"1 / (2 * 7 * 4) = {expected:.6f}")

actual = con.sql(
    "select historical_adjustment_factor from splits "
    "where ticker = 'AAPL' and execution_date = date '2005-02-28'"
).fetchone()[0]
print(f"vendor factor for 2005-02-28 = {actual}")
print("match:", round(expected, 6) == round(actual, 6))

### The boundary is strict

On the execution date, all trading is already adjusted. The join must use
`> D`, not `>= D`. An off-by-one makes one large false return per split.

In [ ]:
# The as-of join, written out.
con.sql('''
    with universe as (
        select ticker, date, close from bars where ticker = 'AAPL'
    )
    select u.date, u.close,
           (select s.historical_adjustment_factor
              from splits s
             where s.ticker = u.ticker
               and s.execution_date > u.date
             order by s.execution_date
             limit 1) as split_factor
    from universe u
    order by u.date
    limit 10
''').pl()

A null factor means no split follows that date — the price needs no adjustment.

---
## 7. Audit invariants on live data

Confirm that the properties the audits enforce at ingest hold on what is published.

In [ ]:
# Split classification: forward > 1, reverse < 1, stock dividend > 1.
con.sql('''
    select adjustment_type,
           count(*) as n,
           min(split_to / split_from) as min_ratio,
           max(split_to / split_from) as max_ratio
    from splits group by adjustment_type order by n desc
''').pl()

In [ ]:
# Reverse splits outnumber forward ~2:1 — mostly distressed microcaps.
con.sql('''
    select adjustment_type, count(*) as n,
           round(100.0 * count(*) / sum(count(*)) over (), 1) as pct
    from splits group by adjustment_type order by n desc
''').pl()

In [ ]:
# RYCEF: factor 0.0. The C shares are not fungible, so no valid adjustment exists.
con.sql('''
    select ticker, execution_date, adjustment_type, split_from, split_to,
           historical_adjustment_factor
    from splits
    where historical_adjustment_factor <= 0
    order by execution_date desc
    limit 10
''').pl()

In [ ]:
# Null dividend factor = vendor had no price on the ex-date.
divs = dal.dividends()
con.sql('''
    select count(*) as rows,
           count(*) filter (historical_adjustment_factor is null) as null_factor,
           round(100.0 * count(*) filter (historical_adjustment_factor is null)
                 / count(*), 1) as pct_null,
           count(*) filter (currency is not null and currency <> 'USD') as not_usd
    from divs
''').pl()

In [ ]:
# Restrict to tickers that trade on the ingested tape.
con.sql('''
    with traded as (select distinct ticker from bars)
    select
        case when d.ticker in (select ticker from traded)
             then 'in day aggs' else 'not in day aggs' end as group_,
        count(*) as rows,
        round(100.0 * count(*) filter (d.historical_adjustment_factor is null)
              / count(*), 1) as pct_null_factor
    from divs d group by 1
''').pl()

~0.97% null inside the traded set vs ~39% outside (foreign issuers, OTC, fund
classes). `adj_close_total` is the default column.

---
## 8. Universe after the backfill

1,255 sessions from 2021-08-23. First look at the models on real history.

In [ ]:
import duckdb
from sdp.config import settings

wh = duckdb.connect(str(settings.warehouse_path), read_only=True)

wh.sql('''
    select date, count(*) filter (in_universe) as names
    from main_staging.stg_universe
    group by 1 order by 1
''').pl()

In [ ]:
# The size of the universe over time. The target was 1,000 to 2,000 names.
wh.sql('''
    with per_date as (
        select date, count(*) filter (in_universe) as n
        from main_staging.stg_universe group by 1
    )
    select date_trunc('year', date) as year,
           round(avg(n)) as avg_names,
           min(n) as min_names,
           max(n) as max_names
    from per_date group by 1 order by 1
''').pl()

~2,960 names at the median, above the 1,000–2,000 target. Tightening
`min_dollar_volume` is a dial. The first 59 sessions have no universe (60-day
history requirement), so the usable window starts 2021-11-15.

In [ ]:
# Where the names are lost — each filter is a column.
wh.sql('''
    select
        count(*)                                  as rows,
        count(*) filter (passes_instrument)       as after_instrument,
        count(*) filter (passes_instrument and passes_price)      as and_price,
        count(*) filter (passes_instrument and passes_price
                         and passes_adv)          as and_adv,
        count(*) filter (in_universe)             as in_universe
    from main_staging.stg_universe
    where date = (select max(date) from main_staging.stg_universe)
''').pl()

---
## 9. Open questions

`python -m sdp.diagnostics` runs these against the whole lake.

### The identifier

Ticker reuse is real. Its size decides whether a simple key works.

In [ ]:
names = dal.tickers()

con.sql('''
    with per_ticker as (
        select ticker, count(distinct composite_figi) as figis
        from names where type = 'CS' and composite_figi is not null
        group by 1
    )
    select count(*) as cs_tickers,
           count(*) filter (figis > 1) as more_than_one_figi,
           count(*) filter (figis > 2) as more_than_two
    from per_ticker
''').pl()

886 of 9,018 CS tickers carry more than one FIGI across the window (one in ten).
The fallback rate after the liquidity screen decides the rule.

In [ ]:
wh.sql('''
    select
        count(*) as cs_rows,
        round(100.0 * count(*) filter (security_key is null
                                       or key_rule <> 'share_class_figi')
              / count(*), 2) as pct_not_on_figi,
        count(*) filter (in_universe) as after_the_screen,
        round(100.0 * count(*) filter (in_universe
                                       and key_rule <> 'share_class_figi')
              / nullif(count(*) filter (in_universe), 0), 2) as pct_not_on_figi_in_universe
    from main_staging.stg_universe
    where type = 'CS'
''').pl()

The screen helps but does not rescue it: 15.4% raw → 9.9% after the screen.
The coalesce with `key_rule` is the answer; report results with and without
the fallback rows.

### The ambiguous event key

In [ ]:
splits = dal.splits()

con.sql('''
    with per_key as (
        select ticker, execution_date,
               count(*) as rows,
               count(distinct historical_adjustment_factor) as factors
        from splits group by 1, 2
    )
    select count(*) as distinct_keys,
           count(*) filter (rows > 1) as duplicated,
           count(*) filter (rows > 1 and factors > 1) as and_disagreeing
    from per_key
''').pl()

209 split keys duplicated (all disagreeing); dividends worse (13,864 duplicated,
7,449 disagreeing). An as-of join against raw rows fans out, so
`stg_corporate_actions` resolves per key first.

In [ ]:
# The trap, demonstrated. Joining prices to the raw splits on the event key
# multiplies rows wherever the key is duplicated.
con.sql('''
    with one_name as (
        select ticker, execution_date
        from splits group by 1, 2 having count(*) > 1 limit 1
    )
    select s.ticker, s.execution_date, s.split_from, s.split_to,
           s.historical_adjustment_factor
    from splits s join one_name using (ticker, execution_date)
''').pl()

---
## 10. Restatement detail

`python -m sdp.restatement` compares the oldest and newest vendor pulls.

In [ ]:
from sdp import restatement

# sdp.restatement reads vendor/, so it keeps working however raw/ is stored.
print(restatement.diff(dal.SPLITS))

Hundreds of ids appear to vanish. Almost all reappear under a new id — the
vendor `id` is not stable across pulls.

In [ ]:
# The event diff, written out, to show what the module does internally.
con.sql('''
    with ka as (select distinct ticker, execution_date from older),
         kb as (select distinct ticker, execution_date from newer)
    select
        (select count(*) from ka where not exists
            (select 1 from kb where kb.ticker = ka.ticker
                                and kb.execution_date = ka.execution_date)) as events_gone,
        (select count(*) from kb where not exists
            (select 1 from ka where ka.ticker = kb.ticker
                                and ka.execution_date = kb.execution_date)) as events_new
''').pl()

5 gone, 82 new on the event key — against 400/479 on the id. The rest was churn.

### Mechanical vs real restatement

A new dividend changes every earlier factor by design. Separate that from the
vendor correcting itself.

In [ ]:
d_pulls = dict(ca.vendor_pulls("massive_dividends"))
d_old, d_new = min(d_pulls), max(d_pulls)
for name, pull in [("d_older", d_old), ("d_newer", d_new)]:
    con.execute(f"""
        create or replace temp view {name} as select * from
        read_json('{d_pulls[pull]}', format='newline_delimited', sample_size=-1)
    """)

con.sql(f'''
    with ua as (select ticker, ex_dividend_date from d_older
                group by 1, 2 having count(*) = 1),
         ub as (select ticker, ex_dividend_date from d_newer
                group by 1, 2 having count(*) = 1),
         k  as (select * from ua intersect select * from ub),
         changed as (
             select a.ticker
             from d_older a join d_newer b using (ticker, ex_dividend_date)
                            join k using (ticker, ex_dividend_date)
             where a.historical_adjustment_factor
                   is distinct from b.historical_adjustment_factor
         ),
         went_ex as (
             select distinct ticker from d_newer
             where ex_dividend_date > date '{d_old}'
               and ex_dividend_date <= date '{d_new}'
         )
    select count(*) as restated,
           count(*) filter (ticker in (select ticker from went_ex)) as mechanical,
           count(*) filter (ticker not in (select ticker from went_ex)) as vendor_revision
    from changed
''').pl()

98.1% mechanical (tickers that went ex inside the window). Only 1,705 rows are
the vendor changing its mind.

---
## 11. Workflow skeleton

A deliberately weak signal (five-day reversal, no residualization) to exercise
the loop and find plumbing errors.

**1. Pull the universe from the model.**

In [ ]:
panel = wh.sql('''
    select u.date, u.security_key, u.ticker, u.close, u.adv,
           p.adj_close_total
    from main_staging.stg_universe u
    join main_staging.stg_prices_adjusted p
      on p.date = u.date and p.ticker = u.ticker
    where u.in_universe
      and u.date >= date '2024-01-01'
''')
print(f"{panel.count('*').fetchone()[0]:,} name-days")
panel.limit(5).pl()

**2. Build a signal.**

In [ ]:
signal = wh.sql('''
    with px as (
        select u.date, u.security_key, p.adj_close_total as px
        from main_staging.stg_universe u
        join main_staging.stg_prices_adjusted p
          on p.date = u.date and p.ticker = u.ticker
        where u.in_universe and p.adj_close_total is not null
    ),
    with_lags as (
        select date, security_key, px,
               lag(px, 5) over w as px_5,
               lead(px, 1) over w as px_next
        from px
        window w as (partition by security_key order by date)
    )
    select date, security_key,
           -- The signal. Negative of the trailing five-day return.
           -(px / px_5 - 1)          as reversal,
           -- The thing it must predict. The next day return.
           px_next / px - 1          as fwd_return
    from with_lags
    where px_5 is not null and px_next is not null
''')
print(f"{signal.count('*').fetchone()[0]:,} rows with a signal and a forward return")

`lead(px, 1)` is the forward return — the only place the future may appear.
Partition by `security_key`, not `ticker`, so a rename does not splice two
companies.

**3. Score it.** Cross-sectional Spearman rank correlation between signal and
forward return, per date, then averaged.

In [ ]:
# `signal` was built on `wh`, and a relation belongs to the connection that
# made it. Querying it from `con` raises.
ic = wh.sql('''
    with daily as (
        select date, corr(rs, rf) as ic
        from (
            select date,
                   rank() over (partition by date order by reversal)   as rs,
                   rank() over (partition by date order by fwd_return) as rf
            from signal
        )
        group by date
        having count(*) > 100
    )
    select count(*)                      as days,
           round(avg(ic), 5)             as mean_ic,
           round(stddev(ic), 5)          as sd_ic,
           round(avg(ic) / (stddev(ic) / sqrt(count(*))), 2) as t_stat
    from daily
''')
ic.pl()

A plumbing check, not a result. Still missing: residualization, purging,
costs, bid-ask bounce. The honest test is whether IC survives a tighter ADV
floor:

```bash
python -m sdp.transform build --vars '{min_dollar_volume: 10000000}'
```

---
## 12. Status

Built: 1,255 sessions of bars and tickers from 2021-08-23, 638 short-volume,
119 short-interest settlements, no gaps; staging models; ~2,960 names/day from
2021-11-15.

Open: universe too loose (tighten `min_dollar_volume`), identifier fallback at
9.9%, coalesce rule to settle.

Next: evaluation harness, PCA risk model, residual reversal.